# Dynamic-Block CSA + HCA in a real LM — WikiText-103 perplexity

Extends `dyn_csa_experiment.py` to a train-from-scratch GPT decoder and compares **perplexity on WikiText-103** across:

| variant | attention per layer |
|---|---|
| `full`          | standard causal self-attention (baseline) |
| `csa_fixed`     | Compressed Sparse Attention, **fixed** blocks (m=4, 2m overlap, top-k) |
| `csa_dynamic`   | CSA with **cosine-similarity dynamic** blocks (ours) |
| `hybrid_fixed`  | interleaved CSA + HCA, **fixed** blocks |
| `hybrid_dynamic`| interleaved CSA + HCA, **dynamic** blocks (ours + HCA) |

**HCA** = Heavily Compressed Attention: heavy non-overlapping compression (m′=64)
+ **dense** attention. **Dynamic** = block boundaries set by cosine similarity.

## Run on Kaggle / Colab (free T4)
1. **Settings → Accelerator → GPU T4 x1**, **Internet ON**.
2. Run All. The first cell installs deps. Defaults fit in ~12–16 GB and run
   ~25–40 min total on a T4 (5 models trained sequentially).


In [ ]:
# Kaggle/Colab already have torch; ensure datasets + tokenizers
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'datasets', 'tokenizers'])
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## Imports & device

In [ ]:
# -*- coding: utf-8 -*-
import math, os, time, json, random
from dataclasses import dataclass, field
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[setup] device = {DEVICE}   torch = {torch.__version__}")


## SECTION 1 — Block-boundary detection (re-used, self-contained here)

In [ ]:
def cosine_similarity_consecutive(H, eps=1e-8):
    h = F.normalize(H, dim=-1, eps=eps)
    return (h[1:] * h[:-1]).sum(dim=-1)

def blocks_from_cuts(n, want_cut_list, min_block, max_block, device):
    """Generate block ids from a list of cut intentions."""
    if n == 0: return torch.empty(0, dtype=torch.long, device=device)
    if n == 1: return torch.zeros(1, dtype=torch.long, device=device)
    
    bids = [0] * n
    cur, cur_len = 0, 1
    pending_cut = False
    for t in range(1, n):
        cur_len += 1
        if t - 1 < len(want_cut_list) and want_cut_list[t - 1]:
            pending_cut = True
        
        may = cur_len > min_block
        # Fix: Change > to >= to strictly prevent length 129 overflowing pos embedding size 128
        must = cur_len >= max_block
        if must or (pending_cut and may):
            cur += 1; cur_len = 1
            pending_cut = False
        bids[t] = cur
    
    bids_t = torch.tensor(bids, dtype=torch.long, device=device)
    return bids_t - bids_t.min()

def blocks_from_cosine(H, tau, min_block, max_block, sim=None):
    n = H.shape[0]; dev = H.device
    if n <= 1: return torch.zeros(n, dtype=torch.long, device=dev)
    if sim is None:
        sim = cosine_similarity_consecutive(H)
    want_cut_list = (sim < tau).cpu().tolist()
    return blocks_from_cuts(n, want_cut_list, min_block, max_block, dev)

def adaptive_threshold(sim, target_block_tokens):
    if sim.numel() == 0:
        return 1.0
    q = max(1.0 / max(target_block_tokens, 1), 1e-3)
    return float(torch.quantile(sim.float().cpu(), q).item())

def blocks_fixed(n, block_size, device):
    return torch.arange(n, device=device) // block_size


## SECTION 2 — Variable-length overlapped softmax-gated pooling (eqs. 9-12)

In [ ]:
def pool_variable_blocks(X, Za, Zb, block_ids, B_pos_a, B_pos_b, overlap):
    n, Fd = X.shape; dev = X.device
    B = int(block_ids.max().item()) + 1
    order = torch.argsort(block_ids, stable=True)
    Xs = X[order]; Zas = Za[order]; Zbs = Zb[order]
    counts = torch.bincount(block_ids, minlength=B)
    starts = torch.cumsum(counts, 0) - counts
    ends = starts + counts
    max_len = int(counts.max().item())
    
    pos = torch.arange(max_len, device=dev)
    g_a = (starts[:, None] + pos[None, :]).clamp(max=n - 1)
    mask_a = (starts[:, None] + pos[None, :]) < ends[:, None]
    Xa = Xs[g_a]; Zab = Zas[g_a]
    
    ov = overlap
    end_prev = ends.roll(1); end_prev[0] = -1
    ov_start = torch.clamp(end_prev - ov, min=0)
    ov_len = end_prev - ov_start
    op = torch.arange(ov, device=dev)
    g_b = (ov_start[:, None] + op[None, :]).clamp(max=n - 1)
    mask_b = op[None, :] < ov_len[:, None]
    Xb = Xs[g_b]; Zbb = Zbs[g_b]
    
    WinX = torch.cat([Xa, Xb], 1); WinZ = torch.cat([Zab, Zbb], 1)
    WinM = torch.cat([mask_a, mask_b], 1)[:, :, None].to(X.dtype)
    WinZ = WinZ.clone()
    
    len_a = min(max_len, B_pos_a.shape[0])
    WinZ[:, :len_a] = WinZ[:, :len_a] + B_pos_a[None, :len_a]
    
    len_b = min(ov, B_pos_b.shape[0])
    if len_b > 0:
        WinZ[:, max_len:max_len + len_b] += B_pos_b[None, :len_b]
    
    sc = F.softmax(WinZ.masked_fill(WinM == 0, float("-inf")), dim=1)
    sc = torch.nan_to_num(sc)
    comp = (sc * WinX * WinM).sum(1)
    last_idx = (starts + counts - 1).clamp(max=n - 1)
    return comp, order[last_idx], B


## SECTION 3 — Lightning Indexer (eqs. 13-17)

In [ ]:
def lightning_indexer(H, comp_kv, last_tok, W_DQ, W_DK, nIH, topk):
    n = H.shape[0]; B = comp_kv.shape[0]; dev = H.device
    hd = W_DQ.shape[1] // nIH
    qI = F.relu((H @ W_DQ).view(n, nIH, hd)).transpose(0, 1)
    kI = F.relu((comp_kv @ W_DK).view(B, nIH, hd)).transpose(0, 1)
    scores = torch.einsum("ind,ibd->inb", qI, kI).sum(0)              # [n, B]
    causal = torch.arange(n, device=dev)[:, None] >= last_tok[None, :]
    scores = scores.masked_fill(~causal, float("-inf"))
    k = min(topk, B)
    
    idx = scores.topk(k, dim=1).indices
    m = torch.zeros(n, B, device=dev, dtype=scores.dtype)
    m.scatter_(1, idx, 1.0)
    m = m * causal.to(m.dtype)
    return m, idx

def _block_token_attn(q, k_blk, v_blk, topk_mask, last_tok, k_sw, v_sw, w, scale):
    n = q.shape[0]; dev = q.device
    K = torch.cat([k_blk, k_sw], 0); V = torch.cat([v_blk, v_sw], 0)
    pos = torch.arange(n, device=dev)
    # Fix: Removed redundant and harmful causal mask multiplication. 
    # topk_mask from lightning_indexer already contains correct causal masking.
    blk_sel = topk_mask
    u = torch.arange(n, device=dev)
    tok_sel = ((u[None, :] <= pos[:, None]) & (u[None, :] > pos[:, None] - w)).to(topk_mask.dtype)
    sel = torch.cat([blk_sel, tok_sel], 1)
    logits = torch.einsum("nhd,mhd->nhm", q, K) * scale
    logits = logits.masked_fill(sel[:, None, :] == 0, float("-inf"))
    attn = torch.softmax(logits, -1); attn = torch.nan_to_num(attn)
    return torch.einsum("nhm,mhd->nhd", attn, V)

def gathered_attention(q, k_blk, v_blk, topk_idx, last_tok, k_sw, v_sw, w, scale, q_chunk=2048):
    n = q.shape[0]; dev = q.device
    out = torch.empty_like(q)
    rel = torch.arange(w, device=dev)
    pos_all = torch.arange(n, device=dev)
    for s in range(0, n, q_chunk):
        e = min(s + q_chunk, n)
        qseg = q[s:e]
        pos = pos_all[s:e]
        ib = topk_idx[s:e]
        kb = k_blk[ib]
        vb = v_blk[ib]
        sel_blk = last_tok[ib] <= pos[:, None]
        win_global = pos[:, None] - (w - 1) + rel[None, :]
        win_valid = win_global >= 0
        wi = win_global.clamp(min=0)
        kw = k_sw[wi]
        vw = v_sw[wi]
        Kset = torch.cat([kb, kw], dim=1)
        Vset = torch.cat([vb, vw], dim=1)
        valid = torch.cat([sel_blk, win_valid], dim=1)
        logits = torch.einsum("qhd,qmhd->qhm", qseg, Kset) * scale
        logits = logits.masked_fill(~valid[:, None, :], torch.finfo(logits.dtype).min)
        attn = torch.softmax(logits, dim=-1)
        attn = torch.nan_to_num(attn)
        out[s:e] = torch.einsum("qhm,qmhd->qhd", attn, Vset)
    return out


## SECTION 4 — Attention layer (full | csa | hca) x (fixed | dynamic)

In [ ]:
@dataclass
class AttnCfg:
    kind: str = "csa"
    chunking: str = "fixed"
    dynamic: bool = False
    block_size: int = 4
    overlap: int = 8          
    index_topk: int = 32
    cos_threshold: float = 0.5
    adaptive: bool = False
    target_block_tokens: int = 16
    temperature: float = 0.1
    min_block: int = 2
    max_block: int = 128      
    sliding_window: int = 128
    c_kv: int = 128
    c_index: int = 64
    n_index_heads: int = 4

class HybridAttention(nn.Module):
    def __init__(self, d_model, n_heads, d_head, cfg: AttnCfg):
        super().__init__()
        self.cfg = cfg
        self.nh, self.hd = n_heads, d_head
        self.kv_dim = 2 * cfg.c_kv
        self.W_KV = nn.Parameter(torch.empty(d_model, self.kv_dim))
        self.W_aZ = nn.Parameter(torch.empty(d_model, self.kv_dim))
        self.W_bZ = nn.Parameter(torch.empty(d_model, self.kv_dim))
        
        self.B_pos_a = nn.Parameter(torch.zeros(128, self.kv_dim))
        self.B_pos_b = nn.Parameter(torch.zeros(32, self.kv_dim))
        
        self.W_q = nn.Linear(d_model, n_heads * d_head, bias=False)
        self.W_kvhead = nn.Linear(self.kv_dim, 2 * n_heads * d_head, bias=False)
        self.W_o = nn.Linear(n_heads * d_head, d_model, bias=False)
        self.W_DQ = nn.Parameter(torch.empty(d_model, cfg.c_index))
        self.W_DK = nn.Parameter(torch.empty(self.kv_dim, cfg.c_index))
        
        Tc = max(cfg.temperature, 1e-3)
        Lb = max(cfg.target_block_tokens, 2)
        delta_init = Tc * math.log(1.0 / (Lb - 1))
        self.delta_logit = nn.Parameter(torch.tensor(float(delta_init)))
        
        # Fix: Removed unused W_gate parameter
        self.last_gate_mean = None
        
        for p in [self.W_KV, self.W_aZ, self.W_bZ, self.W_DQ, self.W_DK]:
            nn.init.xavier_uniform_(p)
        for m in [self.W_q, self.W_kvhead, self.W_o]:
            nn.init.xavier_uniform_(m.weight)
        # Fix: Initialized to None to prevent memory leak during training
        self._stats = None

    def _split(self, kvh):
        nh, hd = self.nh, self.hd
        k = kvh[..., :nh * hd].view(*kvh.shape[:-1], nh, hd)
        v = kvh[..., nh * hd:].view(*kvh.shape[:-1], nh, hd)
        return k, v

    def _single(self, x):
        cfg = self.cfg
        T = x.shape[0]
        scale = 1.0 / math.sqrt(self.hd)
        q = self.W_q(x).view(T, self.nh, self.hd)

        if cfg.kind == "full":
            kvh = self.W_kvhead(x @ self.W_KV)
            k, v = self._split(kvh)
            logits = torch.einsum("nhd,mhd->nhm", q, k) * scale
            mask = torch.triu(torch.full((T, T), float("-inf"), device=x.device), 1)
            attn = torch.softmax(logits + mask[:, None, :], -1)
            out = torch.einsum("nhm,mhd->nhd", attn, v)
            return self.W_o(out.reshape(T, self.nh * self.hd)), None

        KV_raw = x @ self.W_KV
        gate_mean = None
        
        # Fix: HCA kind should not use learnable chunking logic as it performs dense attention
        if cfg.chunking == "cosine_learnable" and cfg.kind != "hca":
            sim = cosine_similarity_consecutive(x)
            if sim.numel() == 0:
                tau = torch.tensor(0.0, device=x.device)
                gate = torch.tensor([], device=x.device)
            else:
                tau = sim.mean() + self.delta_logit
                gate = torch.sigmoid((tau - sim) / max(cfg.temperature, 1e-3))
            gate_mean = gate.mean() if gate.numel() else torch.zeros((), device=x.device)
            
            with torch.no_grad():
                want_cut = (gate.detach() > 0.5).cpu().tolist()
                bid = blocks_from_cuts(T, want_cut, cfg.min_block, cfg.max_block, x.device)
        elif cfg.chunking in ("cosine_abs", "cosine_adaptive") or (cfg.dynamic and cfg.kind != "hca"):
            sim = cosine_similarity_consecutive(x)
            tau = (adaptive_threshold(sim, cfg.target_block_tokens)
                   if (cfg.chunking == "cosine_adaptive" or cfg.adaptive)
                   else cfg.cos_threshold)
            bid = blocks_from_cosine(x, tau, cfg.min_block, cfg.max_block, sim=sim)
        else:
            bid = blocks_fixed(T, cfg.block_size, x.device)
            
        Za = x @ self.W_aZ; Zb = x @ self.W_bZ
        comp_kv, last_tok, Bn = pool_variable_blocks(
            KV_raw, Za, Zb, bid, self.B_pos_a, self.B_pos_b, cfg.overlap)
        
        if self._stats is not None:
            self._stats.append(Bn)

        k_blk, v_blk = self._split(self.W_kvhead(comp_kv))
        k_sw, v_sw = self._split(self.W_kvhead(KV_raw))
        
        if cfg.kind == "hca":
            topk_idx = torch.arange(Bn, device=x.device).unsqueeze(0).expand(T, Bn)
            if T > 1024:
                out = gathered_attention(q, k_blk, v_blk, topk_idx, last_tok,
                                         k_sw, v_sw, cfg.sliding_window, scale)
            else:
                topk_mask = torch.ones(T, Bn, device=x.device)
                out = _block_token_attn(q, k_blk, v_blk, topk_mask, last_tok,
                                        k_sw, v_sw, cfg.sliding_window, scale)
        else:
            topk = cfg.index_topk
            topk_mask, topk_idx = lightning_indexer(x, comp_kv, last_tok, self.W_DQ,
                                                     self.W_DK, cfg.n_index_heads, topk)
            if T > 1024:
                out = gathered_attention(q, k_blk, v_blk, topk_idx, last_tok,
                                         k_sw, v_sw, cfg.sliding_window, scale)
            else:
                out = _block_token_attn(q, k_blk, v_blk, topk_mask, last_tok,
                                        k_sw, v_sw, cfg.sliding_window, scale)
        return self.W_o(out.reshape(T, self.nh * self.hd)), gate_mean

    def forward(self, x):
        outs, gates = [], []
        for b in range(x.shape[0]):
            o, g = self._single(x[b])
            outs.append(o)
            if g is not None:
                gates.append(g)
        self.last_gate_mean = (torch.stack(gates).mean() if gates else None)
        return torch.stack(outs, 0)


## SECTION 5 — Transformer decoder

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.g

class MLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.fc1 = nn.Linear(d, hidden, bias=False)
        self.fc2 = nn.Linear(hidden, d, bias=False)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class Block(nn.Module):
    def __init__(self, d, n_heads, d_head, cfg: AttnCfg, mlp_ratio=4):
        super().__init__()
        self.n1 = RMSNorm(d); self.attn = HybridAttention(d, n_heads, d_head, cfg)
        self.n2 = RMSNorm(d); self.mlp = MLP(d, int(d * mlp_ratio))
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.mlp(self.n2(x))
        return x

class SmallGPT(nn.Module):
    def __init__(self, vocab, d, n_layers, n_heads, d_head, max_seq, layer_cfgs):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(max_seq, d)
        self.blocks = nn.ModuleList([Block(d, n_heads, d_head, c) for c in layer_cfgs])
        self.norm = RMSNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        nn.init.normal_(self.tok.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)
        nn.init.normal_(self.head.weight, std=0.02)
        self.max_seq = max_seq
    def forward(self, ids):
        T = ids.shape[1]
        x = self.tok(ids) + self.pos(torch.arange(T, device=ids.device))
        for blk in self.blocks:
            x = blk(x)
        # Fix: Replaced naive gate_mean sum with a regularized MSE towards target_block_tokens.
        # This provides meaningful gradients to delta_logit instead of just pushing it to 0.
        reg = torch.zeros((), device=ids.device)
        for blk in self.blocks:
            gm = blk.attn.last_gate_mean
            if gm is not None:
                target_gate = 1.0 / max(blk.attn.cfg.target_block_tokens, 1)
                reg = reg + (gm - target_gate) ** 2
        self.comp_reg = reg
        return self.head(self.norm(x))


## SECTION 6 — Layer configs for each variant

In [ ]:
def make_layer_cfgs(n_layers, variant):
    cfgs = []
    for i in range(n_layers):
        if variant == "full":
            cfgs.append(AttnCfg(kind="full", dynamic=False))
        elif variant == "csa_fixed":
            cfgs.append(AttnCfg(kind="csa", dynamic=False, block_size=4, overlap=8, index_topk=32))
        elif variant == "csa_dynamic":
            cfgs.append(AttnCfg(kind="csa", dynamic=True, chunking="cosine_learnable",
                                target_block_tokens=4, overlap=8, index_topk=32, temperature=0.1))
        elif variant == "hybrid_fixed":
            if i % 2 == 0:
                cfgs.append(AttnCfg(kind="csa", dynamic=False, block_size=4, overlap=8, index_topk=32))
            else:
                cfgs.append(AttnCfg(kind="hca", dynamic=False, block_size=64, overlap=8))
        elif variant == "hybrid_dynamic":
            if i % 2 == 0:
                cfgs.append(AttnCfg(kind="csa", dynamic=True, chunking="cosine_learnable",
                                    target_block_tokens=4, overlap=8, index_topk=32, temperature=0.1))
            else:
                cfgs.append(AttnCfg(kind="hca", dynamic=True, chunking="cosine_learnable",
                                    target_block_tokens=64, overlap=8, temperature=0.1))
        else:
            raise ValueError(variant)
    return cfgs


## SECTION 7 — WikiText-103 data (tokenise with a small trained BPE)

In [ ]:
def build_tokenizer(texts, vocab_size=8192):
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import ByteLevel
    tok = Tokenizer(BPE(unk_token="<unk>"))
    tok.pre_tokenizer = ByteLevel(add_prefix_space=True)
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=[
        "<pad>", "<unk>", "<bos>", "<eos>"])
    tok.train_from_iterator(texts, trainer=trainer)
    return tok

def load_wikitext(seq_len, n_train_tokens, vocab_size=8192, val_seqs=512,
                  cache_dir="./wt103_cache"):
    import gc
    from datasets import load_dataset
    os.makedirs(cache_dir, exist_ok=True)
    last_err = None
    for repo in ("Salesforce/wikitext", "wikitext"):
        try:
            ds = load_dataset(repo, "wikitext-103-raw-v1", cache_dir=cache_dir)
            break
        except Exception as e:
            last_err = e
    else:
        raise last_err

    fit_rows = min(len(ds["train"]), 50_000)
    fit_text = ds["train"].select(range(fit_rows))["text"]
    tok = build_tokenizer(fit_text, vocab_size)
    vocab = tok.get_vocab_size()
    del fit_text; gc.collect()

    eos_id = tok.token_to_id("<eos>")

    def stream_tokenise(split, cap_tokens, label):
        out = []
        chunk = 10_000
        for start in range(0, len(ds[split]), chunk):
            rows = ds[split][start:start + chunk]["text"]
            for r in rows:
                if r.strip():
                    out.extend(tok.encode(r).ids)
                    # Fix: Append <eos> to prevent cross-document attention contamination
                    out.append(eos_id)
            if len(out) >= cap_tokens:
                break
            if start and start % 50_000 == 0:
                print(f"    [{label}] {start} rows, {len(out)} tokens")
        return np.array(out[:cap_tokens], dtype=np.int64)

    print(f"[data] tokenising train (cap {n_train_tokens} tokens) ...")
    train_ids = stream_tokenise("train", n_train_tokens + seq_len, "train")
    print(f"[data] train tokens = {len(train_ids)}  vocab = {vocab}")
    print(f"[data] tokenising validation ...")
    val_ids = stream_tokenise("validation", val_seqs * seq_len + seq_len, "val")
    n_val = len(val_ids) // seq_len
    val_batch = val_ids[:n_val * seq_len].reshape(n_val, seq_len)
    print(f"[data] val sequences = {n_val} (seq_len={seq_len})")
    del ds; gc.collect()
    return train_ids, val_batch, vocab, tok

def batch_iter(train_ids, seq_len, batch_size, device, seed=0):
    rng = np.random.default_rng(seed)
    n = len(train_ids) - seq_len - 1
    while True:
        starts = rng.integers(0, n, size=batch_size)
        ids = np.stack([train_ids[s:s + seq_len + 1] for s in starts])
        ids = torch.from_numpy(ids).to(device)
        yield ids[:, :-1], ids[:, 1:]


## SECTION 8 — Train / evaluate

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

@torch.no_grad()
def eval_ppl(model, val_batch, device, chunk=64):
    model.eval()
    nll, ntok = 0.0, 0
    for i in range(0, val_batch.shape[0], chunk):
        ids = torch.from_numpy(val_batch[i:i + chunk]).to(device)
        logits = model(ids)
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                               ids[:, 1:].reshape(-1), reduction="sum")
        nll += loss.item(); ntok += ids[:, 1:].numel()
    # Fix: Removed unnecessary model.train() because the model has no dropout/batchnorm
    return math.exp(nll / max(ntok, 1))

def enable_block_stats(model, on):
    for blk in model.blocks:
        blk.attn._stats = [] if on else None

@torch.no_grad()
def compression_report(model, val_batch, device, n_sample=8):
    report = {}
    seq_len = val_batch.shape[1]
    enable_block_stats(model, True)
    ids = torch.from_numpy(val_batch[:n_sample]).to(device)
    _ = model(ids)
    for li, blk in enumerate(model.blocks):
        attn = blk.attn
        st = attn._stats
        kind = attn.cfg.kind
        if st is not None:
            if len(st) == 0:
                mean_blocks = 1.0
            else:
                mean_blocks = float(np.mean(st))
            report[f"L{li}_{kind}"] = {"blocks": mean_blocks,
                                       "avg_len": seq_len / max(mean_blocks, 1.0)}
            if attn.cfg.dynamic:
                report[f"L{li}_{kind}"]["delta"] = float(attn.delta_logit.item())
    enable_block_stats(model, False)
    return report

def train_variant(variant, train_ids, val_batch, vocab, *, d=256, n_layers=6,
                  n_heads=8, d_head=32, seq_len=512, batch_size=12, steps=500,
                  lr=3e-4, weight_decay=0.1, warmup=50, comp_lambda=0.05,
                  device=DEVICE, log_every=100):
    torch.manual_seed(0)
    cfgs = make_layer_cfgs(n_layers, variant)
    model = SmallGPT(vocab, d, n_layers, n_heads, d_head, seq_len, cfgs).to(device)
    n_param = count_params(model)
    print(f"\n[{variant}] params={n_param/1e6:.2f}M  layers={[c.kind for c in cfgs]}")
    
    decay_params = []
    no_decay_params = []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if any(k in n for k in ["delta_logit", "B_pos_a", "B_pos_b", "W_aZ", "W_bZ"]):
            no_decay_params.append(p)
        else:
            decay_params.append(p)
            
    opt = torch.optim.AdamW([
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ], lr=lr)
    
    bpe = batch_iter(train_ids, seq_len, batch_size, device)
    t0 = time.time(); losses = []
    model.train()
    for step in range(steps):
        x, y = next(bpe)
        if step < warmup:
            for g in opt.param_groups:
                g["lr"] = lr * (step + 1) / warmup
        logits = model(x)
        ce = F.cross_entropy(logits.reshape(-1, vocab), y.reshape(-1))
        loss = ce + comp_lambda * model.comp_reg
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
        if step % log_every == 0 or step == steps - 1:
            print(f"  step {step:4d}  loss {loss.item():.4f}  "
                  f"lr {opt.param_groups[0]['lr']:.2e}  "
                  f"({(time.time()-t0)/max(step+1,1)*1000:.0f}ms/step)")
    ppl = eval_ppl(model, val_batch, device)
    stats = compression_report(model, val_batch, device)
    print(f"[{variant}] val PPL = {ppl:.3f}   comp={stats}")
    del model, opt
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {"ppl": ppl, "params": n_param, "losses": losses, "stats": stats}


## SECTION 9 — Driver

In [ ]:
def run(outdir="results_lm", variants=("full", "csa_fixed", "csa_dynamic",
                                        "hybrid_fixed", "hybrid_dynamic"),
        seq_len=512, batch_size=12, n_train_tokens=4_000_000, steps=500,
        comp_lambda=0.05):
    os.makedirs(outdir, exist_ok=True)
    train_ids, val_batch, vocab, tok = load_wikitext(
        seq_len, n_train_tokens, vocab_size=8192)
    summary = {}
    for v in variants:
        try:
            summary[v] = train_variant(v, train_ids, val_batch, vocab,
                                       seq_len=seq_len, batch_size=batch_size,
                                       steps=steps, comp_lambda=comp_lambda)
        except Exception as e:
            import traceback; summary[v] = {"error": traceback.format_exc()}
            print(f"[{v}] FAILED: {e}")
        with open(os.path.join(outdir, "summary.json"), "w") as f:
            json.dump(summary, f, indent=2, default=float)
    _plot(summary, outdir)
    _print_table(summary)
    return summary

def _print_table(summary):
    print("\n================ WikiText-103 validation perplexity ================")
    print(f"{'variant':16s} {'PPL':>9s} {'params(M)':>10s}")
    for v, s in summary.items():
        if "ppl" in s:
            print(f"{v:16s} {s['ppl']:9.3f} {s['params']/1e6:10.2f}")
    print("====================================================================")

def _plot(summary, outdir):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    items = [(v, s) for v, s in summary.items() if "ppl" in s]
    if not items:
        return
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    vs = [v for v, _ in items]; ps = [s["ppl"] for _, s in items]
    ax[0].bar(range(len(vs)), ps, color="steelblue")
    ax[0].set_xticks(range(len(vs))); ax[0].set_xticklabels(vs, rotation=30, ha="right")
    ax[0].set_ylabel("validation perplexity"); ax[0].set_title("WikiText-103 PPL")
    ax[0].grid(alpha=.3, axis="y")
    for v, s in items:
        if "losses" in s:
            ax[1].plot(s["losses"], label=v, lw=1)
    ax[1].set_xlabel("step"); ax[1].set_ylabel("train loss"); ax[1].set_title("Training curves")
    ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "wikitext_ppl.png"), dpi=120)
    plt.close(fig)

def experiment_blocklen_vs_context(train_ids, vocab, *,
        context_lengths=(2048, 4096, 8192, 16384, 32768),
        variants=("csa_dynamic", "hybrid_dynamic"),
        d=96, n_layers=4, n_heads=4, d_head=24,
        batch_size=1, steps=50, comp_lambda=0.1, lr=3e-4,
        device=DEVICE, outdir="results_lm"):
    os.makedirs(outdir, exist_ok=True)
    results = {v: {"L": [], "mean_len": [], "per_layer": []} for v in variants}
    results["fixed_ref"] = 4.0
    warmup = 10
    for L in context_lengths:
        print(f"\n[long-ctx] L = {L}")
        for v in variants:
            torch.manual_seed(0)
            cfgs = make_layer_cfgs(n_layers, v)
            model = SmallGPT(vocab, d, n_layers, n_heads, d_head, L, cfgs).to(device)
            
            # Fix: Added requires_grad check for no_decay_params to prevent optimizer crashes
            decay_params = [p for n, p in model.named_parameters() if p.requires_grad and "delta_logit" not in n and "B_pos_a" not in n and "B_pos_b" not in n and "W_aZ" not in n and "W_bZ" not in n]
            no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and ("delta_logit" in n or "B_pos_a" in n or "B_pos_b" in n or "W_aZ" in n or "W_bZ" in n)]
            opt = torch.optim.AdamW([
                {'params': decay_params, 'weight_decay': 0.1},
                {'params': no_decay_params, 'weight_decay': 0.0}
            ], lr=lr)
            
            bpe = batch_iter(train_ids, L, batch_size, device)
            model.train(); t0 = time.time()
            for step in range(steps):
                x, y = next(bpe)
                if step < warmup:
                    for g in opt.param_groups:
                        g["lr"] = lr * (step + 1) / warmup
                logits = model(x)
                loss = (F.cross_entropy(logits.reshape(-1, vocab), y.reshape(-1))
                        + comp_lambda * model.comp_reg)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            
            enable_block_stats(model, True)
            nval = 4
            eval_offset = max(0, len(train_ids) - nval * L - L)
            starts = list(range(eval_offset, eval_offset + nval * L, max(L, 1)))[:nval]
            seqs = torch.from_numpy(
                np.stack([train_ids[s:s + L] for s in starts])).to(device)
            with torch.no_grad():
                _ = model(seqs)
            per_layer = {}
            for li, blk in enumerate(model.blocks):
                st = blk.attn._stats
                if st:
                    mb = float(np.mean(st))
                    per_layer[f"L{li}_{blk.attn.cfg.kind}"] = L / max(mb, 1.0)
            enable_block_stats(model, False)
            mean_len = float(np.mean(list(per_layer.values()))) if per_layer else 0.0
            results[v]["L"].append(L)
            results[v]["mean_len"].append(mean_len)
            results[v]["per_layer"].append(per_layer)
            print(f"  [{v:14s}] mean_block_len = {mean_len:6.1f}  "
                  f"per_layer={ {k: round(val,1) for k,val in per_layer.items()} }  "
                  f"({time.time()-t0:.0f}s)")
            del model, opt
            if device.type == "cuda":
                torch.cuda.empty_cache()
    _plot_blocklen_vs_context(results, outdir)
    with open(os.path.join(outdir, "blocklen_vs_context.json"), "w") as f:
        json.dump(results, f, indent=2, default=float)
    return results

def _plot_blocklen_vs_context(results, outdir):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7.5, 5))
    for v, dat in results.items():
        if v == "fixed_ref" or not dat.get("L"):
            continue
        ax.plot(dat["L"], dat["mean_len"], "o-", label=v, lw=2)
    if "fixed_ref" in results:
        ax.axhline(results["fixed_ref"], ls="--", color="gray", label="fixed (m=4)")
    ax.set_xscale("log")
    ax.set_xlabel("context length (tokens, log scale)")
    ax.set_ylabel("learned average block length (tokens)")
    ax.set_title("Learned block length  vs  context length")
    ax.legend(); ax.grid(alpha=.3, which="both")
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "blocklen_vs_context.png"), dpi=120)
    plt.close(fig)


## SECTION 10 — Self-test (no WikiText needed; random tokens)

In [ ]:
def selftest():
    print("=== self-test: forward/backward on random tokens (all kinds) ===")
    torch.manual_seed(0)
    V = 1000; d = 64; nl, nh, hd, T, B = 4, 4, 16, 64, 2
    for variant in ["full", "csa_fixed", "csa_dynamic", "hybrid_fixed", "hybrid_dynamic"]:
        cfgs = make_layer_cfgs(nl, variant)
        m = SmallGPT(V, d, nl, nh, hd, T, cfgs)
        ids = torch.randint(0, V, (B, T))
        logits = m(ids)
        assert logits.shape == (B, T, V), logits.shape
        loss = F.cross_entropy(logits.reshape(-1, V), ids.reshape(-1))
        loss.backward()
        g = sum(p.grad.abs().sum().item() for p in m.parameters() if p.grad is not None)
        print(f"  {variant:14s} logits {tuple(logits.shape)}  loss {loss.item():.3f}  "
              f"grad_sum {g:.2f}  OK")
    print("  SELF-TEST PASSED")


if __name__ == "__main__":
    selftest()


## Run the experiment
Trains all 5 variants on WikiText-103 and prints + plots perplexity. Tune `steps`, `n_train_tokens`, `batch_size` to fit your time budget.

In [ ]:
# self-test first (no data download needed)
selftest()

# full experiment: WikiText-103 perplexity
import os
outdir = '/kaggle/working' if os.path.isdir('/kaggle/working') else 'results_lm'
summary = run(outdir=outdir,
              variants=('full','csa_fixed','csa_dynamic',
                         'hybrid_fixed','hybrid_dynamic'),
              seq_len=512, batch_size=12,
              n_train_tokens=4_000_000, steps=500)

# quick variant: for a 5-minute smoke test, uncomment:
# summary = run(outdir=outdir, variants=('full','csa_dynamic'),
#               seq_len=256, batch_size=8, n_train_tokens=1_000_000, steps=120)


## Long context: learned block length vs context length (8K-32K)

Trains a fresh **learnable** (`csa_dynamic` / `hybrid_dynamic`) model at each
context length and records the **learned average block length per layer**, then
plots block length vs context length. Uses the memory-efficient *gathered* sparse
attention (each query attends only to its top-k blocks + sliding window), so
8K-32K fits on a free T4. `comp_lambda` is the compression budget the block
length is learned against. Drop `32768` from the list to run faster.

In [ ]:
# load ~1M tokens (enough headroom to sample 32K windows)
train_ids, val_batch, vocab, _ = load_wikitext(
    seq_len=2048, n_train_tokens=1_000_000, vocab_size=8192)

ctx_exp = experiment_blocklen_vs_context(
    train_ids, vocab,
    context_lengths=(2048, 4096, 8192, 16384, 32768),
    variants=('csa_dynamic', 'hybrid_dynamic'),
    d=96, n_layers=4, n_heads=4, d_head=24,
    batch_size=1, steps=50, comp_lambda=0.1,
    outdir=outdir)
# faster smoke version (no 32K):
# ctx_exp = experiment_blocklen_vs_context(
#     train_ids, vocab, context_lengths=(2048, 4096, 8192),
#     variants=('csa_dynamic',), steps=15, outdir=outdir)
